In [2]:
!pip install langchain langchain-community langchain-groq
!pip install pymysql sqlalchemy python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_NAME = os.getenv("DB_NAME")

print(f"Groq API Key:{GROQ_API_KEY[-4:]}")
print(f"DB: {DB_USER}|{DB_HOST}:{DB_PORT}|{DB_NAME}")


TypeError: 'NoneType' object is not subscriptable

In [7]:
from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit, create_sql_agent

db = SQLDatabase.from_uri(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    include_tables=[
        "assets", "daily_prices", "macro_indicators",
        "etf_flows", "energy_mining", "market_events",
    ],
    sample_rows_in_table_info=2,
)

print("Connected to MySQL")
print("Tables:", db.get_usable_table_names())

ModuleNotFoundError: No module named 'langchain_groq'

In [ ]:
SYSTEM_PROMPT = """
You are a gold market analyst. You have access to the gold_market MySQL database.

Tables (exact column names):
assets: asset_id (PK), symbol, name, category, data_source
daily_prices: price_id (PK), asset_id (FK to assets.asset_id), price_date, close_price, open_price, high_price, low_price, volume
macro_indicators: macro_id (PK), indicator_date, fed_rate, treasury_10y, treasury_2y, real_rate_calc, yield_curve_10y2y, cpi_index, inflation_yoy, unemployment, m2_supply
etf_flows: asset_id (FK to assets.asset_id), flow_date, etf_price, gld_gold_premium, etf_signal
energy_mining: energy_id (PK), record_date, wti_oil, brent_oil, copper_price, mining_cost_index, gold_mining_margin, mining_bull_signal
market_events: event_id (PK), event_start, event_end, event_type, event_name, region, impact_on_gold

Asset symbols in assets table:
GC=F (Gold), SI=F (Silver), ^GSPC (S&P 500), DX-Y.NYB (DXY), ^VIX (VIX),
CL=F (WTI Oil), BZ=F (Brent Oil), HG=F (Copper), GDX (Gold Miners ETF),
GLD (SPDR Gold ETF), IAU (iShares Gold ETF), ^TNX (10Y Treasury),
^IRX (3M Treasury), TIPS (iShares TIPS ETF), RINF (Inflation ETF)

Ready-to-use views (prefer these over raw table joins):
v_gold_dxy_fed: price_date, gold_price, dxy_index, fed_rate, treasury_10y, inflation_yoy, real_rate_calc, yield_curve_10y2y
v_gold_during_events: event_name, event_type, region, impact_on_gold, event_start, event_end, price_date, gold_price, sp500_price
v_macro_predictors: indicator_date, fed_rate, treasury_10y, treasury_2y, real_rate_calc, yield_curve_10y2y, inflation_yoy, unemployment, m2_supply, gold_price, gold_return_1m, gold_up5pct_signal
v_asset_correlation: price_date, gold_price, silver_price, dxy_index, sp500_price, vix_index
v_energy_mining_crisis: record_date, wti_oil, brent_oil, copper_price, mining_cost_index, gold_mining_margin, mining_bull_signal, gold_price, event_name, event_type, impact_on_gold

Stored procedures:
CALL sp_gold_return_period('2020-01-01', '2020-12-31') - returns: period_start, period_end, gold_price_start, gold_price_end, absolute_return, return_pct
CALL sp_macro_regime_analysis(2.0) - returns: regime, trading_days, avg_gold_price, min_gold_price, max_gold_price, avg_inflation_yoy, avg_real_rate
CALL sp_mining_margin_history('2020-01-01', '2023-12-31') - returns: record_date, wti_oil, copper_price, mining_cost_index, gold_mining_margin, mining_bull_signal, gold_price, margin_30d_avg

Join rules (when writing raw SQL):
1. Gold = symbol 'GC=F' in assets
2. daily_prices - macro_indicators: price_date = indicator_date
3. daily_prices - energy_mining: price_date = record_date
4. daily_prices - market_events: price_date BETWEEN event_start AND COALESCE(event_end, CURDATE())

Output rules:
- Round prices to 2 decimal places, yields/rates to 4 decimal places
- Prefer VIEWs and stored procedures over writing raw JOINs from scratch
- If the question matches a stored procedure exactly - use CALL, not SELECT

STRICT RULES:
1. Schema is fully provided above - NEVER call sql_db_list_tables or sql_db_schema.
2. Call sql_db_query_checker EXACTLY ONCE per query, then immediately call sql_db_query.
3. Never repeat sql_db_query_checker on the same query.
4. To join daily_prices with assets always use: asset_id = (SELECT asset_id FROM assets WHERE symbol = 'GC=F')
"""

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, api_key=GROQ_API_KEY)

class _QueryOnlyToolkit(SQLDatabaseToolkit):
    """Strips schema-exploration tools - schema is provided in the system prompt."""
    def get_tools(self):
        return [t for t in super().get_tools()
                if t.name in ("sql_db_query", "sql_db_query_checker")]

agent = create_sql_agent(
    llm=llm,
    toolkit=_QueryOnlyToolkit(db=db, llm=llm),
    agent_type="zero-shot-react-description",
    verbose=True,
    prefix=SYSTEM_PROMPT,
    max_iterations=5,
    handle_parsing_errors=True,
)

print("Done")

In [28]:
def ask(question):
    result = agent.invoke({"input": question})
    print(result["output"])

---
# Prompts

In [ ]:
ask("During which crises did gold prices rise the most? Show the top 5 events by price increase.")

In [29]:
ask("Show the average price of gold, silver and the S&P 500 by year starting in 2010")



> Entering new SQL Agent Executor chain...
Thought: I should first list the tables in the database to see what I can query.
Action: sql_db_list_tables
Action Input: assets, daily_prices, energy_mining, etf_flows, macro_indicators, market_eventsThought: I now know the tables in the database. I can see that the daily_prices table seems to be the most relevant for this query, as it contains the daily prices of the assets. I should query the schema of this table to see what fields are available.
Action: sql_db_schema
Action Input: daily_prices
CREATE TABLE daily_prices (
	price_id INTEGER NOT NULL AUTO_INCREMENT, 
	asset_id INTEGER NOT NULL, 
	price_date DATE NOT NULL, 
	close_price DECIMAL(15, 4), 
	PRIMARY KEY (price_id), 
	CONSTRAINT daily_prices_ibfk_1 FOREIGN KEY(asset_id) REFERENCES assets (asset_id)
)DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_unicode_ci ENGINE=InnoDB

/*
2 rows from daily_prices table:
price_id	asset_id	price_date	close_price
1	5	2000-01-03	100.2200
2	6	2000-01-03	14

In [ ]:
ask("What is the average price of gold when the real interest rate is negative (real_rate_calc < 0)?")

In [ ]:
ask("")